# 06b-c supplement - Branch-ELM 8.002 parametri sui dati HayFlow

ATTENZIONE: questa revisione e ritirata. Il completamento matched usava il vecchio H2+refit e un contratto informativo diverso da Branch-ELM. Non eseguire finche il notebook non viene sostituito dalla correzione con il sistema compatto 06b e input causali identici.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})
raise RuntimeError('Notebook 06b-c Branch-ELM ritirato: attende la correzione con il sistema compatto 06b e un contratto informativo identico.')

## 1. Dataset e fresh test

La precedente lista di input H2/05j-n apparteneva al confronto ritirato e non costituisce il contratto corretto.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source
from src.hayflow_model.hines_frozen_candidate_micro_rollout import EXPECTED_05JO_INDEX_SHA256
from src.hayflow_model.hines_regenerative_fresh_test import EXPECTED_05JN_INDEX_SHA256
from src.hayflow_model import EXPECTED_BRANCH_ELM_RESUME_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
fresh_override=os.environ.get('HAYFLOW_05JO_ARTIFACT');FRESH_TEST_SOURCE=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_05JO_INDEX_SHA256,override=Path(fresh_override) if fresh_override else None);assert FRESH_TEST_SOURCE is not None,'Artefatto hayflow_hines_regenerative_fresh_test esatto non trovato.'
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06bc_elm_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.'
h2_override=os.environ.get('HAYFLOW_05B_ARTIFACT');h2_candidates=([Path(h2_override).expanduser()] if h2_override else [])+list(INPUT_ROOT.rglob('hayflow_hines_canary_v2.zip'))+[p.parent.parent for p in INPUT_ROOT.rglob('canary_models.pt') if p.parent.name=='checkpoints'];H2_SOURCE=next((p.resolve() for p in h2_candidates if p.exists()),None);assert H2_SOURCE is not None,'Checkpoint hayflow_hines_canary_v2 non trovato.'
refit_override=os.environ.get('HAYFLOW_05JN_ARTIFACT');REFIT_SOURCE=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_05JN_INDEX_SHA256,override=Path(refit_override) if refit_override else None);assert REFIT_SOURCE is not None,'Artefatto hayflow_hines_regenerative_decoder_refit esatto non trovato.'
resume_override=os.environ.get('HAYFLOW_BRANCH_ELM_RESUME');RESUME_SOURCE=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_BRANCH_ELM_RESUME_INDEX_SHA256,override=Path(resume_override) if resume_override else None)
print({'fresh_test':str(FRESH_TEST_SOURCE),'base':str(BASE_SOURCE),'composite_manifest':str(COMPOSITE_MANIFEST),'frozen_h2':str(H2_SOURCE),'frozen_refit':str(REFIT_SOURCE),'elm_resume':None if RESUME_SOURCE is None else str(RESUME_SOURCE)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow ELM][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880;print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Contratto ELM e training ridotto

Il preflight deve confermare esattamente 8.002 parametri e 639+639 canali. Le traiettorie con corrente somatica vengono escluse e conteggiate, non adattate artificialmente. Il primo preflight ha identificato 48 episodi compatibili: prima di qualsiasi training o lettura degli outcome fresh, i ruoli sono stati corretti a 28 fit, 10 calibration e 10 development.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import BranchELMEnrichedBenchmark,BranchELMEnrichedBenchmarkConfig,restore_registered_branch_elm_checkpoints
cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/branch_elm_enriched_benchmark.yml').read_text());config=BranchELMEnrichedBenchmarkConfig.from_mapping(cfg['branch_elm_enriched_benchmark'])
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_branch_elm_enriched_benchmark')
resume_existing=OUTPUT_DIR.exists()
if not resume_existing and RESUME_SOURCE is not None:
 resume_report=restore_registered_branch_elm_checkpoints(RESUME_SOURCE,OUTPUT_DIR,'/kaggle/working/.06bc_elm_resume_cache');display({'checkpoint_resume':resume_report['valid'],'checkpoint_count':resume_report['checkpoint_count'],'retraining':resume_report['retraining_performed']});assert resume_report['valid'] and resume_report['checkpoint_count']==6;resume_existing=True
session=BranchELMEnrichedBenchmark(bundle,OUTPUT_DIR,config,FRESH_TEST_SOURCE,ELM_REPO,code_revision=REVISION,resume_existing_output=resume_existing);contract=session.prepare_benchmark();display({'valid':contract['valid'],'sidecar_only':contract['sidecar_only'],'parameters':contract['trainable_parameter_count'],'dendritic_sites':contract['dendritic_segment_count'],'roles':contract['roles'],'exclusions':contract['exclusions'],'fresh_episodes':contract['fresh_test_episode_count'],'fresh_compatible':contract['fresh_test_compatible_episode_count'],'input_views':contract['input_views'],'resume_existing':resume_existing});assert contract['valid'] and contract['trainable_parameter_count']==8002 and contract['sidecar_only'] and contract['fresh_test_used_for_selection'] is False

In [ ]:
try:
 results=session.run_benchmark();matched=session.run_matched_hayflow_comparison(H2_SOURCE,REFIT_SOURCE,results);final_report=session.finalize(contract,results,matched)
finally:
 session.close()
zero={view:{'soma_rmse_mv':round(row['clipped_soma_rmse_mv'],4),'AUC':None if row['spike_auc'] is None else round(row['spike_auc'],4),'episodes':row['episode_count']} for view,row in final_report['zero_shot_original_checkpoint'].items()}
display({'valid':final_report['valid'],'architecture':final_report['architecture'],'elm_parameters':final_report['trainable_parameter_count'],'published_original_soma_rmse_mv':final_report['published_original_dataset_reference']['clipped_soma_rmse_mv'],'zero_shot_fresh':zero,'elm_retrained_fresh_median':{k:round(v,4) for k,v in final_report['fresh_test_median_retrained_clipped_soma_rmse_mv'].items()},'matched_hayflow_median_soma_rmse_mv':round(matched['frozen_hayflow']['median_clipped_soma_rmse_mv'],4),'matched_hayflow_ensemble_soma_rmse_mv':round(matched['frozen_hayflow']['ensemble_mean']['clipped_soma_rmse_mv'],4),'matched_transition_count':matched['metric_contract']['evaluated_transition_count'],'same_metric':matched['metric_contract']['same_metric'],'same_transitions':matched['metric_contract']['same_transitions'],'same_input_contract':matched['metric_contract']['same_input_contract'],'median_hayflow_gain_vs_elm_realized':round(matched['median_hayflow_error_reduction_vs_branch_elm_fraction'],4),'voltage_comparison_complete':matched['comparison_complete_for_voltage'],'spike_comparison_complete':matched['comparison_complete_for_spikes']});assert final_report['valid'] and matched['comparison_complete_for_voltage'] and not matched['frozen_hayflow']['retraining_performed'] and final_report['primary_experiment_replaced'] is False and final_report['fresh_test_used_for_selection'] is False

## 3. Crea e scarica lo ZIP

Usa il downloader browser stabile del progetto; non stampa checkpoint o array estesi.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_branch_elm_enriched_benchmark','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})